In [ ]:
from pathlib import Path
import io
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# 1 - localizar o CSV geral (localmente) ou fazer upload (no Colab)
caminhos_locais = [
    Path("../dados/artigos_scopus_G.csv"),
    Path("dados/artigos_scopus_G.csv"),
]
arquivo_local = next((p for p in caminhos_locais if p.exists()), None)

if arquivo_local is not None:
    dados_brutos = pd.read_csv(arquivo_local, encoding="utf-8-sig")
    print("Lido localmente:", arquivo_local)
else:
    from google.colab import files
    arquivos = files.upload()
    nome = next(iter(arquivos))
    dados_brutos = pd.read_csv(io.BytesIO(arquivos[nome]), encoding="utf-8-sig")

# 2 - filtrar apenas Article e Review (mesmo critério metodológico de Odeilson/analise.py)
TIPOS = ["Article", "Review"]
dados = dados_brutos[dados_brutos["tipo_doc"].isin(TIPOS)].copy()

# 3 - conferir a base
print("Total de documentos no CSV:", len(dados_brutos))
print(f"Total após filtrar {TIPOS}:", len(dados))
print("Quantidade de colunas:", len(dados.columns))

# 4 - mostrar as primeiras três publicações
display(dados[[
    "titulo", "ano", "paises", "fund_sponsor"
]].head(3))

Lido localmente: ..\dados\artigos_scopus_G.csv
Total de documentos no CSV: 2154
Total após filtrar ['Article', 'Review']: 1840
Quantidade de colunas: 15
                                              titulo  ...                                   fund_sponsor
1  Bionic porous konjac glucomannan/polylactic ac...  ...  Natural Science Foundation of Fujian Province
2  Thermoplastic Polyurethane-Based Composite Fil...  ...   National Natural Science Foundation of China
3  Cooling and mechanical performance of a novel ...  ...   National Natural Science Foundation of China

[3 rows x 4 columns]


In [2]:
# 1 - identificar os artigos com financiamento declarado
# (considera qualquer campo de financiamento preenchido: sponsor, sigla ou número)

dados["tem_financiamento"] = (
    dados[["fund_sponsor", "fund_acr", "fund_no"]]
    .fillna("")
    .astype(str)
    .apply(lambda col: col.str.strip().ne(""))
    .any(axis=1)
)

# 2 - contagem dos resultados
total = len(dados)
com_financiamento = int(dados["tem_financiamento"].sum())
sem_financiamento = total - com_financiamento

# 3 - calculo de porcentagem
percentual = com_financiamento / total * 100

print("Total de artigos:", total)
print("Com financiamento declarado:", com_financiamento)
print("Sem financiamento declarado:", sem_financiamento)
print(f"Percentual: {percentual:.1f}%")

Total de artigos: 1840
Com financiamento declarado: 1561
Sem financiamento declarado: 279
Percentual: 84.8%


In [3]:
# 1 - função para transformar a coluna 'paises' (separada por ';') em lista
def extrair_paises(paises_str):
    if pd.isna(paises_str):
        return []
    paises = []
    for pais in str(paises_str).split(";"):
        pais = pais.strip()
        if pais and pais not in paises:
            paises.append(pais)
    return sorted(paises)

# 2 - aplicar a função a todos (preservando dados["paises"] original)
dados["paises_lista"] = dados["paises"].apply(extrair_paises)

display(dados[["titulo", "paises_lista"]].head(5))

contagem = dados.explode("paises_lista")["paises_lista"].value_counts()

print("Dez valores mais frequentes:")
print(contagem.head(10))

print("Artigos sem país identificado:",
      int((dados["paises_lista"].str.len() == 0).sum()))

                                              titulo        paises_lista
1  Bionic porous konjac glucomannan/polylactic ac...             [China]
2  Thermoplastic Polyurethane-Based Composite Fil...             [China]
3  Cooling and mechanical performance of a novel ...  [China, Singapore]
4  Multifunctional Face Mask with Daytime Passive...             [China]
5  Hybrid Directional Radiative Cooling and Solar...             [China]
Dez valores mais frequentes:
paises_lista
China             1250
United States      228
Hong Kong          154
South Korea        112
United Kingdom      65
Australia           55
India               54
Singapore           41
Spain               36
Italy               33
Name: count, dtype: int64
Artigos sem país identificado: 1


In [4]:
# listar os países identificados
todos_paises = dados.explode("paises_lista")["paises_lista"].value_counts()

print("Quantidade de nomes diferentes:", len(todos_paises))

# mostrar os valores menos frequentes para auditoria
print("Valores menos frequentes:")
display(todos_paises[todos_paises <= 2])

Quantidade de nomes diferentes: 63
Valores menos frequentes:
paises_lista
Lebanon             2
Tunisia             1
Kazakhstan          1
Rwanda              1
Hungary             1
Austria             1
New Zealand         1
Jordan              1
Iraq                1
Malta               1
Ecuador             1
Macao               1
Argentina           1
Mali                1
Mongolia            1
Ukraine             1
Croatia             1
French Polynesia    1
Name: count, dtype: int64


In [5]:
# corrigir afiliações mal formatadas (caso exista alguma inconsistência pontual)
# Obs.: na base artigos_scopus_G.csv da API da Scopus esses registros já vêm estruturados

correcoes = {
    "2-s2.0-86000432474": ["China", "Singapore", "United Kingdom"],
    "2-s2.0-85180259058": ["China"],
    "2-s2.0-85084329567": ["Morocco"],
    "2-s2.0-85129693896": ["China"],
}

# preservar a classificação original
dados["paises_corrigidos"] = dados["paises_lista"].copy()

# aplicar as correções (ordenadas para manter padrão consistente)
for eid, paises in correcoes.items():
    linhas = dados.index[dados["eid"] == eid]
    if len(linhas) == 1:
        dados.at[linhas[0], "paises_corrigidos"] = sorted(paises)
    else:
        print("Verificar artigo:", eid)

# conferir
contagem_corrigida = (
    dados.explode("paises_corrigidos")
    ["paises_corrigidos"]
    .value_counts()
)

print("Países e territórios diferentes:", len(contagem_corrigida))

print("\nDez mais frequentes:")
print(contagem_corrigida.head(10))

print("\nArtigos sem país identificado:",
      int((dados["paises_corrigidos"].str.len() == 0).sum()))

Países e territórios diferentes: 63

Dez mais frequentes:
paises_corrigidos
China             1250
United States      228
Hong Kong          154
South Korea        112
United Kingdom      65
Australia           55
India               54
Singapore           41
Spain               36
Italy               33
Name: count, dtype: int64

Artigos sem país identificado: 1


In [6]:
# separar por país
base_paises = dados.explode("paises_corrigidos")

# retirar linhas sem país identificado
base_paises = base_paises.dropna(
    subset=["paises_corrigidos"]
)

# agrupar os registros
resultado = base_paises.groupby("paises_corrigidos").agg(
    total_artigos=("eid", "nunique"),
    com_financiamento=("tem_financiamento", "sum")
)

# calcular a porcentagem de financiamento
resultado["percentual"] = (
    resultado["com_financiamento"]
    / resultado["total_artigos"] * 100
).round(1)

# ordenar pelo número de artigos
resultado = resultado.sort_values(
    "total_artigos", ascending=False
)

# mostrar os dez países com mais publicações
display(resultado.head(10))

                   total_artigos  com_financiamento  percentual
paises_corrigidos                                              
China                       1250               1130        90.4
United States                228                180        78.9
Hong Kong                    154                150        97.4
South Korea                  112                107        95.5
United Kingdom                65                 61        93.8
Australia                     55                 46        83.6
India                         54                 22        40.7
Singapore                     41                 39        95.1
Spain                         36                 34        94.4
Italy                         33                 25        75.8


In [7]:
import matplotlib.pyplot as plt

# selecionar países com mais artigos
top10 = resultado.head(10)

# organizar pelo percentual
top10 = top10.sort_values("percentual")

# criar o gráfico
fig, ax = plt.subplots(figsize=(10, 6))

barras = ax.barh(
    top10.index,
    top10["percentual"]
)

# adicionar as porcentagens e quantidades
for barra, (_, linha) in zip(barras, top10.iterrows()):
    ax.text(
        linha["percentual"] + 1,
        barra.get_y() + barra.get_height() / 2,
        f'{linha["percentual"]:.1f}% (n={linha["total_artigos"]:.0f})',
        va="center",
        fontsize=9
    )

# identificar o gráfico
ax.set_xlabel("Artigos com financiamento declarado (%)")
ax.set_title(
    "Financiamento declarado por país\n"
    "Scopus (busca G), 2000–2025"
)

ax.set_xlim(0, 115)

plt.tight_layout()

# salvar a imagem
plt.savefig(
    "grafico_financiamento_pais.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [8]:
# conferir artigos associados a China
china_antes = dados["paises_lista"].apply(
    lambda lista: "China" in lista
)

china_depois = dados["paises_corrigidos"].apply(
    lambda lista: "China" in lista
)

print("China antes da correção:", int(china_antes.sum()))
print("China depois da correção:", int(china_depois.sum()))

# identificar os registros que mudaram
mudancas = dados.loc[
    china_antes != china_depois,
    ["eid", "titulo", "paises_lista", "paises_corrigidos"]
]

print("\nArtigos cuja classificação mudou:")
display(mudancas)

# conferir duplicados
print("EIDs duplicados:", int(dados["eid"].duplicated().sum()))

China antes da correção: 1250
China depois da correção: 1250

Artigos cuja classificação mudou:
Empty DataFrame
Columns: [eid, titulo, paises_lista, paises_corrigidos]
Index: []
EIDs duplicados: 0


In [9]:
import matplotlib.pyplot as plt

# selecionar os 10 países
top10 = resultado.head(10).copy()

# ordenar pelo percentual
top10 = top10.sort_values("percentual")

# criar grafico
fig, ax = plt.subplots(figsize=(10, 6))

barras = ax.barh(
    top10.index,
    top10["percentual"]
)

for barra, (_, linha) in zip(barras, top10.iterrows()):
    ax.text(
        linha["percentual"] + 1,
        barra.get_y() + barra.get_height() / 2,
        f'{linha["percentual"]:.1f}%',
        va="center"
    )

ax.set_xlim(0, 112)

ax.set_xlabel("Artigos com financiamento declarado (%)")
ax.set_ylabel("País ou território")

ax.set_title(
    "Financiamento declarado por país ou território\n"
    "Resfriamento radiativo passivo | 2000–2025"
)

plt.tight_layout()

plt.savefig(
    "grafico_financiamento.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [10]:
# exportar tabela completa por país
resultado.to_csv(
    "financiamento_por_pais.csv",
    encoding="utf-8-sig"
)

print("Tabela salva com sucesso!")

Tabela salva com sucesso!


In [11]:
# artigos publicados entre 2015 e 2025
base_recente = base_paises[
    base_paises["ano"].astype(int).between(2015, 2025)
].copy()

# calcular financiamento
resultado_recente = base_recente.groupby(
    "paises_corrigidos"
).agg(
    total_artigos=("eid", "nunique"),
    com_financiamento=("tem_financiamento", "sum")
)

resultado_recente["percentual"] = (
    resultado_recente["com_financiamento"]
    / resultado_recente["total_artigos"] * 100
).round(1)

# comparar com os dez países mais produtivos
paises_top10 = resultado.head(10).index

comparacao = pd.DataFrame({
    "2000-2025 (%)": resultado.loc[
        paises_top10, "percentual"
    ],
    "2015-2025 (%)": resultado_recente.reindex(
        paises_top10
    )["percentual"]
})

display(comparacao)

                   2000-2025 (%)  2015-2025 (%)
paises_corrigidos                              
China                       90.4           90.7
United States               78.9           80.9
Hong Kong                   97.4           97.4
South Korea                 95.5           95.5
United Kingdom              93.8           93.8
Australia                   83.6           86.8
India                       40.7           40.7
Singapore                   95.1           95.1
Spain                       94.4           94.3
Italy                       75.8           75.8


In [12]:
top10 = resultado.head(10).index

tabela_final = pd.DataFrame({
    "Total 2000-2025": resultado.loc[
        top10, "total_artigos"
    ],
    "Financiados 2000-2025": resultado.loc[
        top10, "com_financiamento"
    ],
    "Percentual 2000-2025": resultado.loc[
        top10, "percentual"
    ],
    "Total 2015-2025": resultado_recente.reindex(
        top10
    )["total_artigos"],
    "Financiados 2015-2025": resultado_recente.reindex(
        top10
    )["com_financiamento"],
    "Percentual 2015-2025": resultado_recente.reindex(
        top10
    )["percentual"]
})

display(tabela_final)

tabela_final.to_csv(
    "comparacao_financiamento_periodos.csv",
    encoding="utf-8-sig"
)

print("Tabela comparativa salva!")

                   Total 2000-2025  ...  Percentual 2015-2025
paises_corrigidos                   ...                      
China                         1250  ...                  90.7
United States                  228  ...                  80.9
Hong Kong                      154  ...                  97.4
South Korea                    112  ...                  95.5
United Kingdom                  65  ...                  93.8
Australia                       55  ...                  86.8
India                           54  ...                  40.7
Singapore                       41  ...                  95.1
Spain                           36  ...                  94.3
Italy                           33  ...                  75.8

[10 rows x 6 columns]
Tabela comparativa salva!


In [13]:
tabela_final = resultado.head(10).copy()

tabela_final = tabela_final.rename(columns={
    "total_artigos": "Total de artigos",
    "com_financiamento": "Com financiamento declarado",
    "percentual": "Percentual (%)"
})

tabela_final.index.name = "País ou território"

display(tabela_final)

tabela_final.to_csv(
    "financiamento_por_pais_final.csv",
    encoding="utf-8-sig"
)

print("Tabela final salva!")

                    Total de artigos  ...  Percentual (%)
País ou território                    ...                
China                           1250  ...            90.4
United States                    228  ...            78.9
Hong Kong                        154  ...            97.4
South Korea                      112  ...            95.5
United Kingdom                    65  ...            93.8
Australia                         55  ...            83.6
India                             54  ...            40.7
Singapore                         41  ...            95.1
Spain                             36  ...            94.4
Italy                             33  ...            75.8

[10 rows x 3 columns]
Tabela final salva!


In [14]:
import sys

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("financiamento_por_pais_final.csv")
    files.download("grafico_financiamento.png")
    files.download("comparacao_financiamento_periodos.csv")
else:
    print("Execução local concluída: arquivos salvos na pasta Gabriel/.")

Execução local concluída: arquivos salvos na pasta Gabriel/.


### Análise do financiamento declarado por país

**Metodologia**

A partir da base unificada do grupo (`dados/artigos_scopus_G.csv`, busca ampliada na Scopus entre 2000 e 2025, com 2.154 registros no total), foram analisados **1.840** documentos dos tipos *Article* e *Review* (mesmo recorte geográfico utilizado por Odeilson). Os dados foram processados em Python com as bibliotecas `pandas` e `matplotlib`.

Os países e territórios foram identificados a partir do campo estruturado `paises` extraído pela API da Scopus (mantendo Hong Kong separado da China, conforme a base). Publicações com instituições de diferentes países foram contabilizadas em cada país correspondente (contagem inteira).

Para calcular a proporção de financiamento declarado, considerou-se financiado todo artigo com pelo menos um dos campos de fomento preenchido (`fund_sponsor`, `fund_acr` ou `fund_no`), dividindo o total de financiados pelo total de artigos associados a cada país e multiplicando por 100.

**Resultados**

Dos **1.840** artigos analisados, **1.561** apresentaram informações de financiamento declarado, correspondendo a **84,8%** da base (279 sem registro de financiamento).

A China apresentou o maior volume de publicações, com **1.250** registros, dos quais **90,4%** (1.130) possuíam financiamento declarado. Os Estados Unidos apresentaram **78,9%** (180 de 228), enquanto Hong Kong alcançou **97,4%** (150 de 154) e a Coreia do Sul **95,5%** (107 de 112).

Entre os dez países e territórios mais produtivos, Hong Kong (**97,4%**), Coreia do Sul (**95,5%**) e Singapura (**95,1%**) apresentaram os maiores percentuais de financiamento declarado, enquanto a Índia apresentou o menor percentual, de **40,7%** (22 de 54).

A comparação complementar com o período **2015–2025** indicou estabilidade ou pequenas altas na maioria dos países analisados (China de 90,4% para 90,7%; Estados Unidos de 78,9% para 80,9%; Austrália de 83,6% para 86,8%).

**Limitações**

O indicador não representa o volume de recursos financeiros investidos, mas a presença de metadados de financiamento nos registros da Scopus. A ausência dessa informação não significa necessariamente ausência de apoio financeiro.

A cobertura histórica dos metadados de financiamento pode influenciar as comparações entre períodos. Além disso, a atribuição de artigos com colaboração internacional a mais de um país produz contagens não exclusivas.